# Cross-Dataset Evaluation: UTD-MHAD → MSR-Action3D

Test UTD-MHAD deployment models (trained on all UTD-MHAD data, depth+skeleton only)
on **unseen MSR-Action3D data** using overlapping action classes.

**7 matching classes:**

| Action | UTD-MHAD class | MSR-Action3D class |
|---|---|---|
| Draw X | 8. draw_X | 7. draw_x |
| Draw circle | 9. draw_circle_CW + 10. CCW (merged) | 9. draw_circle |
| Clap | 4. clap | 10. hand_clap |
| Tennis swing | 15. tennis_swing | 17. tennis_swing |
| Tennis serve | 17. tennis_serve | 18. tennis_serve |
| Jog | 22. jog | 16. jogging |
| Pickup & throw | 21. pickup_and_throw | 20. pickup_throw |

**Pipeline:** Load MSR-Action3D features → filter to 7 matching classes → remap to
merged UTD labels → apply independent scaler → logit-mask model output to matching
classes only → report accuracy & timing per method.

In [70]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [71]:
import numpy as np
import pandas as pd
import joblib
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 4090


## Configuration & Paths

In [72]:
# UTD-MHAD deployment artifacts (no-video model, depth+skeleton only)
UTD_RESULTS_ROOT = Path("results_all_skel_depth")

# MSR-Action3D extracted features
MSR_FEATURE_DIR = Path("../msr-action3d/features")

# Output directory
CROSS_RESULTS = Path("results_cross_dataset_msr")
CROSS_RESULTS.mkdir(exist_ok=True)

# All 14 metaheuristic method names
META_METHODS = [
    'meta_BAT', 'meta_CS', 'meta_DE', 'meta_FFA', 'meta_GA', 'meta_GWO',
    'meta_HHO', 'meta_JAYA', 'meta_MFO', 'meta_MVO', 'meta_PSO',
    'meta_SCA', 'meta_SSA', 'meta_WOA'
]

USE_CUDA_SYNC = torch.cuda.is_available()

def sync_cuda():
    if USE_CUDA_SYNC:
        torch.cuda.synchronize()

print(f"UTD-MHAD results: {UTD_RESULTS_ROOT}")
print(f"MSR-Action3D features: {MSR_FEATURE_DIR}")
print(f"Output: {CROSS_RESULTS}")

UTD-MHAD results: results_all_skel_depth
MSR-Action3D features: ../msr-action3d/features
Output: results_cross_dataset_msr


## Model Definition (must match UTD-MHAD training)

In [73]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    def forward(self, x):
        return self.classifier(x)

print("Model defined (matches UTD-MHAD deployment)")

Model defined (matches UTD-MHAD deployment)


## Load UTD-MHAD Artifacts & Class Info

In [74]:
# Load UTD-MHAD label encoder
utd_le = joblib.load(Path("features_new") / "label_encoder.pkl")
utd_classes = list(utd_le.classes_)
UTD_NUM_CLASSES = len(utd_classes)
print(f"UTD-MHAD: {UTD_NUM_CLASSES} classes")
print(f"UTD classes (encoded→raw): {list(enumerate(utd_classes))}")

# UTD-MHAD encoded indices for matching classes (using raw action numbers)
utd_draw_x_idx    = utd_classes.index(8)     # Draw X
utd_draw_cw_idx   = utd_classes.index(9)     # Draw circle CW
utd_draw_ccw_idx  = utd_classes.index(10)    # Draw circle CCW
utd_clap_idx      = utd_classes.index(4)     # Clap
utd_tennis_sw_idx = utd_classes.index(15)    # Tennis swing
utd_tennis_sv_idx = utd_classes.index(17)    # Tennis serve
utd_jog_idx       = utd_classes.index(22)    # Jog
utd_pickup_idx    = utd_classes.index(21)    # Pickup and throw

print(f"\nUTD matching class indices:")
print(f"  Draw X:          {utd_draw_x_idx}")
print(f"  Draw circle CW:  {utd_draw_cw_idx}")
print(f"  Draw circle CCW: {utd_draw_ccw_idx}")
print(f"  Clap:            {utd_clap_idx}")
print(f"  Tennis swing:    {utd_tennis_sw_idx}")
print(f"  Tennis serve:    {utd_tennis_sv_idx}")
print(f"  Jog:             {utd_jog_idx}")
print(f"  Pickup & throw:  {utd_pickup_idx}")

UTD-MHAD: 27 classes
UTD classes (encoded→raw): [(0, np.int64(1)), (1, np.int64(2)), (2, np.int64(3)), (3, np.int64(4)), (4, np.int64(5)), (5, np.int64(6)), (6, np.int64(7)), (7, np.int64(8)), (8, np.int64(9)), (9, np.int64(10)), (10, np.int64(11)), (11, np.int64(12)), (12, np.int64(13)), (13, np.int64(14)), (14, np.int64(15)), (15, np.int64(16)), (16, np.int64(17)), (17, np.int64(18)), (18, np.int64(19)), (19, np.int64(20)), (20, np.int64(21)), (21, np.int64(22)), (22, np.int64(23)), (23, np.int64(24)), (24, np.int64(25)), (25, np.int64(26)), (26, np.int64(27))]

UTD matching class indices:
  Draw X:          7
  Draw circle CW:  8
  Draw circle CCW: 9
  Clap:            3
  Tennis swing:    14
  Tennis serve:    16
  Jog:             21
  Pickup & throw:  20


## Load MSR-Action3D Features & Class Info

In [75]:
# Load MSR-Action3D features
msr_X_feat = joblib.load(MSR_FEATURE_DIR / "X_feat.pkl")
msr_y = np.load(MSR_FEATURE_DIR / "y.npy")
msr_subjects = np.load(MSR_FEATURE_DIR / "subjects.npy")
msr_le = joblib.load(MSR_FEATURE_DIR / "label_encoder.pkl")
msr_classes = list(msr_le.classes_)

print(f"MSR-Action3D: {len(msr_X_feat)} samples, {len(msr_classes)} classes, "
      f"{len(np.unique(msr_subjects))} subjects")
print(f"MSR classes: {msr_classes}")

# MSR action names (1-indexed)
MSR_ACTION_NAMES = [
    'high_arm_wave', 'horizontal_arm_wave', 'hammer', 'hand_catch',
    'forward_punch', 'high_throw', 'draw_x', 'draw_tick',
    'draw_circle', 'hand_clap', 'two_hand_wave', 'side_boxing',
    'bend', 'forward_kick', 'side_kick', 'jogging',
    'tennis_swing', 'tennis_serve', 'golf_swing', 'pickup_throw'
]

# Map MSR encoded labels to action names
print("\nMSR-Action3D class index mapping:")
msr_idx_to_name = {}
for enc_idx, raw_label in enumerate(msr_le.classes_):
    action_num = int(raw_label)
    name = MSR_ACTION_NAMES[action_num - 1] if action_num <= len(MSR_ACTION_NAMES) else f"action_{action_num}"
    msr_idx_to_name[enc_idx] = name
    count = np.sum(msr_y == enc_idx)
    print(f"  {enc_idx:2d} (raw {raw_label}): {name:25s} [{count} samples]")

# Verify feature dimensions match UTD-MHAD
msr_first = msr_X_feat[0]
msr_dims = {
    'depth': msr_first['depth_feat'].shape[0] if msr_first['depth_feat'].size > 0 else 0,
    'skeleton': msr_first['skeleton_feat'].shape[0] if msr_first['skeleton_feat'].size > 0 else 0,
}
print(f"\nMSR feature dims: {msr_dims}")

# Load UTD dims for comparison
utd_first = joblib.load(Path("features_new") / "X_feat.pkl")[0]
utd_dims = {
    'depth': utd_first['depth_feat'].shape[0] if utd_first['depth_feat'].size > 0 else 0,
    'skeleton': utd_first['skeleton_feat'].shape[0] if utd_first['skeleton_feat'].size > 0 else 0,
}
UTD_TOTAL_FEATURES = utd_dims['depth'] + utd_dims['skeleton']
print(f"UTD feature dims: {utd_dims}")
print(f"UTD total (depth+skeleton): {UTD_TOTAL_FEATURES}")

# Check compatibility
for mod in ['depth', 'skeleton']:
    assert msr_dims[mod] == utd_dims[mod], (
        f"{mod} feature dimension mismatch! MSR: {msr_dims[mod]} vs UTD: {utd_dims[mod]}")
print("\nFeature dimensions MATCH!")

MSR-Action3D: 567 samples, 20 classes, 10 subjects
MSR classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]

MSR-Action3D class index mapping:
   0 (raw 1): high_arm_wave             [27 samples]
   1 (raw 2): horizontal_arm_wave       [27 samples]
   2 (raw 3): hammer                    [27 samples]
   3 (raw 4): hand_catch                [26 samples]
   4 (raw 5): forward_punch             [26 samples]
   5 (raw 6): high_throw                [26 samples]
   6 (raw 7): draw_x                    [28 samples]
   7 (raw 8): draw_tick                 [30 samples]
   8 (raw 9): draw_circle               [30 samples]
   9 (raw 10): hand_clap                 [30 samples]
  10 (raw 11): two_hand_wave             [30 samples]
  11 (raw 12): side_boxing               [30

## Define Class Mapping (7 matching classes)

In [76]:
# =====================================================================
# Define the 7 matching classes between UTD-MHAD and MSR-Action3D
# =====================================================================

MERGED_LABELS = {
    0: 'draw_x',
    1: 'draw_circle',
    2: 'clap',
    3: 'tennis_swing',
    4: 'tennis_serve',
    5: 'jog',
    6: 'pickup_throw',
}
MERGED_NAMES = [MERGED_LABELS[i] for i in range(len(MERGED_LABELS))]
N_MERGED = len(MERGED_LABELS)

# --- UTD-MHAD model output index → merged label ---
utd_to_merged = {
    utd_draw_x_idx:    0,  # draw_x
    utd_draw_cw_idx:   1,  # draw_circle (CW)
    utd_draw_ccw_idx:  1,  # draw_circle (CCW) — merged with CW
    utd_clap_idx:      2,  # clap
    utd_tennis_sw_idx: 3,  # tennis_swing
    utd_tennis_sv_idx: 4,  # tennis_serve
    utd_jog_idx:       5,  # jog
    utd_pickup_idx:    6,  # pickup_throw
}

# Allowed UTD output indices for logit masking
utd_allowed_indices = sorted(utd_to_merged.keys())

print("UTD-MHAD → Merged label mapping:")
for utd_idx, merged_label in sorted(utd_to_merged.items()):
    print(f"  UTD encoded {utd_idx:2d} (raw {utd_classes[utd_idx]}) → {merged_label} ({MERGED_NAMES[merged_label]})")

# --- MSR-Action3D encoded label → merged label ---
msr_to_merged = {}

for enc_idx, name in msr_idx_to_name.items():
    if name == 'draw_x':
        msr_to_merged[enc_idx] = 0
    elif name == 'draw_circle':
        msr_to_merged[enc_idx] = 1
    elif name == 'hand_clap':
        msr_to_merged[enc_idx] = 2
    elif name == 'tennis_swing':
        msr_to_merged[enc_idx] = 3
    elif name == 'tennis_serve':
        msr_to_merged[enc_idx] = 4
    elif name == 'jogging':
        msr_to_merged[enc_idx] = 5
    elif name == 'pickup_throw':
        msr_to_merged[enc_idx] = 6

print(f"\nMSR-Action3D → Merged label mapping:")
for msr_idx, merged_label in sorted(msr_to_merged.items()):
    print(f"  MSR encoded {msr_idx:2d} ({msr_idx_to_name[msr_idx]:25s}) → {merged_label} ({MERGED_NAMES[merged_label]})")

assert len(utd_to_merged) == 8, f"Expected 8 UTD entries (7 classes, CW+CCW separate), got {len(utd_to_merged)}"
assert len(msr_to_merged) == 7, f"Expected 7 MSR entries, got {len(msr_to_merged)}"
print(f"\nAll 7 merged classes mapped successfully!")

UTD-MHAD → Merged label mapping:
  UTD encoded  3 (raw 4) → 2 (clap)
  UTD encoded  7 (raw 8) → 0 (draw_x)
  UTD encoded  8 (raw 9) → 1 (draw_circle)
  UTD encoded  9 (raw 10) → 1 (draw_circle)
  UTD encoded 14 (raw 15) → 3 (tennis_swing)
  UTD encoded 16 (raw 17) → 4 (tennis_serve)
  UTD encoded 20 (raw 21) → 6 (pickup_throw)
  UTD encoded 21 (raw 22) → 5 (jog)

MSR-Action3D → Merged label mapping:
  MSR encoded  6 (draw_x                   ) → 0 (draw_x)
  MSR encoded  8 (draw_circle              ) → 1 (draw_circle)
  MSR encoded  9 (hand_clap                ) → 2 (clap)
  MSR encoded 15 (jogging                  ) → 5 (jog)
  MSR encoded 16 (tennis_swing             ) → 3 (tennis_swing)
  MSR encoded 17 (tennis_serve             ) → 4 (tennis_serve)
  MSR encoded 19 (pickup_throw             ) → 6 (pickup_throw)

All 7 merged classes mapped successfully!


## Prepare MSR-Action3D Test Data

In [77]:
# Filter MSR-Action3D to only matching classes
matching_msr_indices = sorted(msr_to_merged.keys())
keep_mask = np.isin(msr_y, matching_msr_indices)

msr_X_matched = [msr_X_feat[i] for i in range(len(msr_X_feat)) if keep_mask[i]]
msr_y_matched = msr_y[keep_mask]

print(f"MSR-Action3D samples with matching classes: {len(msr_y_matched)} / {len(msr_y)}")
for idx in matching_msr_indices:
    count = np.sum(msr_y_matched == idx)
    print(f"  MSR {idx} ({msr_idx_to_name[idx]}): {count} samples")

# Filter to only samples that have BOTH depth and skeleton features
msr_X_matched_full = []
msr_y_matched_full = []
for i, sample in enumerate(msr_X_matched):
    has_depth = sample['depth_feat'].size > 0
    has_skel = sample['skeleton_feat'].size > 0
    if has_depth and has_skel:
        msr_X_matched_full.append(sample)
        msr_y_matched_full.append(msr_y_matched[i])
    
dropped = len(msr_X_matched) - len(msr_X_matched_full)
if dropped > 0:
    print(f"Dropped {dropped} samples missing depth or skeleton")

msr_X_matched = msr_X_matched_full
msr_y_matched = np.array(msr_y_matched_full)

# Build unified feature vectors (depth + skeleton, same order as UTD no-video)
msr_X_unified = []
for sample in msr_X_matched:
    feat_vector = np.concatenate([
        sample['depth_feat'],
        sample['skeleton_feat']
    ])
    msr_X_unified.append(feat_vector)
msr_X_unified = np.array(msr_X_unified)

print(f"\nMSR unified feature shape: {msr_X_unified.shape}")
print(f"Expected (UTD total): {UTD_TOTAL_FEATURES}")
assert msr_X_unified.shape[1] == UTD_TOTAL_FEATURES, "Feature dimension mismatch!"

# Independent scaling (fit on MSR data only)
msr_scaler = StandardScaler()
msr_X_scaled = msr_scaler.fit_transform(msr_X_unified)
print("Applied independent StandardScaler to MSR-Action3D features")

# Create merged ground truth labels
msr_y_merged = np.array([msr_to_merged[label] for label in msr_y_matched])

print(f"\nMerged {N_MERGED}-class label distribution:")
for label_id in range(N_MERGED):
    count = np.sum(msr_y_merged == label_id)
    print(f"  {label_id} ({MERGED_NAMES[label_id]}): {count} samples")
print(f"  Total: {len(msr_y_merged)}")

# Remove classes with 0 samples
active_labels = [l for l in range(N_MERGED) if np.sum(msr_y_merged == l) > 0]
inactive_labels = [l for l in range(N_MERGED) if np.sum(msr_y_merged == l) == 0]

if inactive_labels:
    print(f"\nRemoving {len(inactive_labels)} classes with 0 samples: "
          f"{[MERGED_NAMES[l] for l in inactive_labels]}")
    
    # Filter samples to only active classes
    keep_mask = np.isin(msr_y_merged, active_labels)
    msr_X_scaled = msr_X_scaled[keep_mask]
    msr_y_merged = msr_y_merged[keep_mask]
    
    # Remap labels to be contiguous 0..N-1
    old_to_new = {old: new for new, old in enumerate(active_labels)}
    msr_y_merged = np.array([old_to_new[l] for l in msr_y_merged])
    
    # Update merged names and count
    MERGED_NAMES = [MERGED_NAMES[l] for l in active_labels]
    N_MERGED = len(MERGED_NAMES)
    
    # Update UTD mapping to use new labels
    utd_to_merged = {k: old_to_new[v] for k, v in utd_to_merged.items() if v in old_to_new}
    utd_allowed_indices = sorted(utd_to_merged.keys())
    
    print(f"Remaining: {N_MERGED} classes, {len(msr_y_merged)} samples")
    for i in range(N_MERGED):
        print(f"  {i} ({MERGED_NAMES[i]}): {np.sum(msr_y_merged == i)} samples")

MSR-Action3D samples with matching classes: 208 / 567
  MSR 6 (draw_x): 28 samples
  MSR 8 (draw_circle): 30 samples
  MSR 9 (hand_clap): 30 samples
  MSR 15 (jogging): 30 samples
  MSR 16 (tennis_swing): 30 samples
  MSR 17 (tennis_serve): 30 samples
  MSR 19 (pickup_throw): 30 samples
Dropped 97 samples missing depth or skeleton

MSR unified feature shape: (111, 1861)
Expected (UTD total): 1861
Applied independent StandardScaler to MSR-Action3D features

Merged 7-class label distribution:
  0 (draw_x): 28 samples
  1 (draw_circle): 30 samples
  2 (clap): 30 samples
  3 (tennis_swing): 0 samples
  4 (tennis_serve): 0 samples
  5 (jog): 23 samples
  6 (pickup_throw): 0 samples
  Total: 111

Removing 3 classes with 0 samples: ['tennis_swing', 'tennis_serve', 'pickup_throw']
Remaining: 4 classes, 111 samples
  0 (draw_x): 28 samples
  1 (draw_circle): 30 samples
  2 (clap): 30 samples
  3 (jog): 23 samples


## Evaluate All Methods on MSR-Action3D

In [78]:
def load_and_evaluate(method_name, X_test_scaled, total_features, num_classes):
    """
    Load UTD-MHAD deployment mask + model, classify MSR-Action3D data
    with logit masking to restrict output to matching classes only.
    """
    method_dir = UTD_RESULTS_ROOT / method_name

    # Load feature mask
    mask = np.load(method_dir / "deployment_mask.npy")
    n_selected = int(np.sum(mask))

    # Load model
    model = SimpleNN(n_selected, num_classes).to(DEVICE)
    model.load_state_dict(torch.load(method_dir / "deployment_model.pth", map_location=DEVICE))
    model.eval()

    # Logit mask: only allow matching UTD classes
    logit_mask = torch.full((num_classes,), float('-inf'), device=DEVICE)
    for c in utd_allowed_indices:
        logit_mask[c] = 0.0

    # GPU warmup
    dummy = torch.FloatTensor(X_test_scaled[0:1, mask]).to(DEVICE)
    with torch.no_grad():
        for _ in range(50):
            _ = model(dummy)
    sync_cuda()

    # Classify all samples
    raw_preds = []
    mask_times = []
    class_times = []

    for i in range(len(X_test_scaled)):
        sample = X_test_scaled[i:i+1]

        start_mask = time.perf_counter()
        sample_masked = sample[:, mask]
        end_mask = time.perf_counter()

        sample_tensor = torch.FloatTensor(sample_masked).to(DEVICE)
        sync_cuda()

        start_class = time.perf_counter()
        with torch.no_grad():
            output = model(sample_tensor)
            masked_output = output + logit_mask
            pred = torch.argmax(masked_output, dim=1).cpu().item()
        sync_cuda()
        end_class = time.perf_counter()

        raw_preds.append(pred)
        mask_times.append((end_mask - start_mask) * 1000)
        class_times.append((end_class - start_class) * 1000)

    raw_preds = np.array(raw_preds)
    total_times = [m + c for m, c in zip(mask_times, class_times)]

    # Convert UTD predictions to merged labels
    merged_preds = np.array([utd_to_merged.get(p, -1) for p in raw_preds])

    return {
        'raw_preds': raw_preds,
        'merged_preds': merged_preds,
        'n_features': n_selected,
        'feature_retention_pct': n_selected / total_features * 100,
        'avg_mask_time_ms': np.mean(mask_times),
        'avg_class_time_ms': np.mean(class_times),
        'avg_total_time_ms': np.mean(total_times),
        'std_total_time_ms': np.std(total_times),
    }

print("Evaluation function defined")

Evaluation function defined


In [79]:
# Run evaluation for baseline + all 14 metaheuristics
all_methods = ['baseline'] + META_METHODS
results = {}

print("=" * 90)
print("CROSS-DATASET EVALUATION: UTD-MHAD → MSR-Action3D (7-class)")
print("=" * 90)

for method in all_methods:
    print(f"\n  Evaluating {method}...")

    res = load_and_evaluate(method, msr_X_scaled, UTD_TOTAL_FEATURES, UTD_NUM_CLASSES)

    acc = accuracy_score(msr_y_merged, res['merged_preds'])
    f1 = f1_score(msr_y_merged, res['merged_preds'], average='macro', zero_division=0)
    res['accuracy'] = acc
    res['f1_macro'] = f1
    results[method] = res

    outside = np.sum(res['merged_preds'] == -1)
    print(f"    Acc={acc*100:.2f}%, F1={f1*100:.2f}%, "
          f"Features: {res['n_features']}/{UTD_TOTAL_FEATURES} "
          f"({res['feature_retention_pct']:.1f}%), "
          f"Avg inference: {res['avg_total_time_ms']:.4f} ms")

print("\n" + "=" * 90)
print("ALL EVALUATIONS COMPLETE")
print("=" * 90)

CROSS-DATASET EVALUATION: UTD-MHAD → MSR-Action3D (7-class)

  Evaluating baseline...
    Acc=42.34%, F1=39.55%, Features: 1861/1861 (100.0%), Avg inference: 0.1826 ms

  Evaluating meta_BAT...
    Acc=46.85%, F1=44.93%, Features: 960/1861 (51.6%), Avg inference: 0.2634 ms

  Evaluating meta_CS...
    Acc=50.45%, F1=49.20%, Features: 886/1861 (47.6%), Avg inference: 0.1604 ms

  Evaluating meta_DE...
    Acc=44.14%, F1=42.00%, Features: 929/1861 (49.9%), Avg inference: 0.1522 ms

  Evaluating meta_FFA...
    Acc=40.54%, F1=36.36%, Features: 915/1861 (49.2%), Avg inference: 0.1595 ms

  Evaluating meta_GA...
    Acc=46.85%, F1=47.26%, Features: 897/1861 (48.2%), Avg inference: 0.1505 ms

  Evaluating meta_GWO...
    Acc=46.85%, F1=44.55%, Features: 770/1861 (41.4%), Avg inference: 0.1499 ms

  Evaluating meta_HHO...
    Acc=49.55%, F1=50.70%, Features: 1041/1861 (55.9%), Avg inference: 0.1668 ms

  Evaluating meta_JAYA...
    Acc=53.15%, F1=52.87%, Features: 900/1861 (48.4%), Avg infere

## Results Summary

In [80]:
# Identify best metaheuristic
best_meta = max(META_METHODS, key=lambda m: results[m]['accuracy'])
best_meta_res = results[best_meta]
baseline_res = results['baseline']

# Build results table
rows = []
for method in all_methods:
    r = results[method]
    rows.append({
        'Method': method,
        'Accuracy (%)': r['accuracy'] * 100,
        'F1 Macro (%)': r['f1_macro'] * 100,
        'Features': r['n_features'],
        'Feature Retention (%)': r['feature_retention_pct'],
        'Mask Time (ms)': r['avg_mask_time_ms'],
        'Class Time (ms)': r['avg_class_time_ms'],
        'Total Time (ms)': r['avg_total_time_ms'],
    })

df_results = pd.DataFrame(rows).round(4)
df_results = df_results.sort_values('Accuracy (%)', ascending=False)

print("\n" + "=" * 100)
print("CROSS-DATASET RESULTS: UTD-MHAD → MSR-Action3D (7-class)")
print("=" * 100)
print(df_results.to_string(index=False))

df_results.to_csv(CROSS_RESULTS / "cross_dataset_results_msr.csv", index=False)
print(f"\nSaved: {CROSS_RESULTS / 'cross_dataset_results_msr.csv'}")


CROSS-DATASET RESULTS: UTD-MHAD → MSR-Action3D (7-class)
   Method  Accuracy (%)  F1 Macro (%)  Features  Feature Retention (%)  Mask Time (ms)  Class Time (ms)  Total Time (ms)
 meta_WOA       57.6577       59.1441       465                24.9866          0.0087           0.1543           0.1631
 meta_MVO       55.8559       55.1834       926                49.7582          0.0083           0.1404           0.1487
meta_JAYA       53.1532       52.8661       900                48.3611          0.0109           0.1520           0.1629
 meta_MFO       50.4505       49.9271       889                47.7700          0.0081           0.1432           0.1514
  meta_CS       50.4505       49.1952       886                47.6088          0.0088           0.1516           0.1604
 meta_HHO       49.5495       50.6967      1041                55.9377          0.0087           0.1580           0.1668
 meta_BAT       46.8468       44.9317       960                51.5852          0.0087         

## Best Metaheuristic — Detailed Analysis

In [81]:
print("=" * 85)
print("BEST METAHEURISTIC FOR CROSS-DATASET TRANSFER (4-class)")
print("=" * 85)

speedup = baseline_res['avg_total_time_ms'] / best_meta_res['avg_total_time_ms'] if best_meta_res['avg_total_time_ms'] > 0 else float('inf')
acc_diff = best_meta_res['accuracy'] - baseline_res['accuracy']

print(f"\nBest metaheuristic: {best_meta}")
print(f"  Accuracy:          {best_meta_res['accuracy']*100:.2f}%")
print(f"  F1 Macro:          {best_meta_res['f1_macro']*100:.2f}%")
print(f"  Features:          {best_meta_res['n_features']}/{UTD_TOTAL_FEATURES} "
      f"({best_meta_res['feature_retention_pct']:.1f}%)")
print(f"  Avg total time:    {best_meta_res['avg_total_time_ms']:.4f} ms")

print(f"\nBaseline (all features):")
print(f"  Accuracy:          {baseline_res['accuracy']*100:.2f}%")
print(f"  F1 Macro:          {baseline_res['f1_macro']*100:.2f}%")
print(f"  Avg total time:    {baseline_res['avg_total_time_ms']:.4f} ms")

print(f"\nComparison:")
print(f"  Accuracy difference:  {acc_diff*100:+.2f}%")
print(f"  Inference speedup:    {speedup:.2f}x")
print(f"  Feature reduction:    {100 - best_meta_res['feature_retention_pct']:.1f}%")
print(f"  Random chance (4-class): {100/N_MERGED:.1f}%")

BEST METAHEURISTIC FOR CROSS-DATASET TRANSFER (4-class)

Best metaheuristic: meta_WOA
  Accuracy:          57.66%
  F1 Macro:          59.14%
  Features:          465/1861 (25.0%)
  Avg total time:    0.1631 ms

Baseline (all features):
  Accuracy:          42.34%
  F1 Macro:          39.55%
  Avg total time:    0.1826 ms

Comparison:
  Accuracy difference:  +15.32%
  Inference speedup:    1.12x
  Feature reduction:    75.0%
  Random chance (4-class): 25.0%


In [82]:
# Per-class classification report for best metaheuristic
print(f"\nPer-class report ({best_meta}):")
print("-" * 60)
print(classification_report(
    msr_y_merged, results[best_meta]['merged_preds'],
    labels=list(range(N_MERGED)),
    target_names=MERGED_NAMES,
    zero_division=0
))

cm = confusion_matrix(msr_y_merged, results[best_meta]['merged_preds'],
                      labels=list(range(N_MERGED)))
print("Confusion matrix (rows=true, cols=pred):")
cm_df = pd.DataFrame(cm, index=MERGED_NAMES, columns=MERGED_NAMES)
print(cm_df)

# Raw UTD prediction distribution
raw = results[best_meta]['raw_preds']
print(f"\nRaw UTD prediction distribution ({best_meta}):")
for cls_idx in sorted(np.unique(raw)):
    count = np.sum(raw == cls_idx)
    name = utd_classes[cls_idx] if cls_idx < len(utd_classes) else f"unknown({cls_idx})"
    is_match = "✓" if cls_idx in utd_to_merged else "✗"
    print(f"  {is_match} UTD {cls_idx:2d} ({name}): {count:4d} samples")


Per-class report (meta_WOA):
------------------------------------------------------------
              precision    recall  f1-score   support

      draw_x       0.57      0.61      0.59        28
 draw_circle       0.41      0.53      0.46        30
        clap       0.67      0.60      0.63        30
         jog       0.87      0.57      0.68        23

    accuracy                           0.58       111
   macro avg       0.63      0.58      0.59       111
weighted avg       0.61      0.58      0.59       111

Confusion matrix (rows=true, cols=pred):
             draw_x  draw_circle  clap  jog
draw_x           17            5     6    0
draw_circle       9           16     3    2
clap              3            9    18    0
jog               1            9     0   13

Raw UTD prediction distribution (meta_WOA):
  ✓ UTD  3 (4):   27 samples
  ✓ UTD  7 (8):   30 samples
  ✓ UTD  8 (9):   19 samples
  ✓ UTD  9 (10):   20 samples
  ✓ UTD 21 (22):   15 samples


In [83]:
# Same for baseline
print(f"\nPer-class report (baseline):")
print("-" * 60)
print(classification_report(
    msr_y_merged, results['baseline']['merged_preds'],
    labels=list(range(N_MERGED)),
    target_names=MERGED_NAMES,
    zero_division=0
))

cm_base = confusion_matrix(msr_y_merged, results['baseline']['merged_preds'],
                           labels=list(range(N_MERGED)))
print("Confusion matrix (rows=true, cols=pred):")
cm_base_df = pd.DataFrame(cm_base, index=MERGED_NAMES, columns=MERGED_NAMES)
print(cm_base_df)


Per-class report (baseline):
------------------------------------------------------------
              precision    recall  f1-score   support

      draw_x       0.29      0.29      0.29        28
 draw_circle       0.48      0.50      0.49        30
        clap       0.57      0.13      0.22        30
         jog       0.44      0.87      0.59        23

    accuracy                           0.42       111
   macro avg       0.45      0.45      0.40       111
weighted avg       0.45      0.42      0.39       111

Confusion matrix (rows=true, cols=pred):
             draw_x  draw_circle  clap  jog
draw_x            8            2     2   16
draw_circle       9           15     1    5
clap             11           11     4    4
jog               0            3     0   20


## Visualization

In [84]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Bar chart: all methods accuracy
fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_results.sort_values('Accuracy (%)', ascending=False)['Method'].tolist()
accs = df_results.set_index('Method').loc[methods_sorted, 'Accuracy (%)']

bars = ax.bar(range(len(methods_sorted)), accs.values, alpha=0.7, color='steelblue')
ax.set_xticks(range(len(methods_sorted)))
ax.set_xticklabels(methods_sorted, rotation=60, ha='right')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Cross-Dataset Accuracy: UTD-MHAD → MSR-Action3D (7-class)', fontsize=14)
ax.axhline(y=100/N_MERGED, color='red', linestyle='--', alpha=0.5,
           label=f'Random baseline ({100/N_MERGED:.1f}%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    val = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., val + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(CROSS_RESULTS / 'cross_dataset_accuracy_msr.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {CROSS_RESULTS / 'cross_dataset_accuracy_msr.png'}")

Saved: results_cross_dataset_msr/cross_dataset_accuracy_msr.png


## Timing Summary

In [86]:
print("\n" + "=" * 85)
print("INFERENCE TIMING SUMMARY (MSR-Action3D cross-dataset, 4-class)")
print("=" * 85)

print(f"""
┌───────────────────────────────────┬──────────────┬──────────────────────┐
│ Component                         │ Baseline     │ {best_meta:<20s} │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Mask Application                  │     ---      │ {best_meta_res['avg_mask_time_ms']:>10.4f} ms        │
│ NN Forward Pass                   │ {baseline_res['avg_class_time_ms']:>10.4f} ms│ {best_meta_res['avg_class_time_ms']:>10.4f} ms        │
│ Total per-sample                  │ {baseline_res['avg_total_time_ms']:>10.4f} ms│ {best_meta_res['avg_total_time_ms']:>10.4f} ms        │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Inference Speedup                 │    1.00x     │ {speedup:>10.2f}x          │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ MSR-Action3D Accuracy (4-class)   │ {baseline_res['accuracy']*100:>9.2f}%   │ {best_meta_res['accuracy']*100:>9.2f}%           │
│ Random Chance (4-class)           │ {100/N_MERGED:>9.1f}%   │ {100/N_MERGED:>9.1f}%           │
│ Features Used                     │ {UTD_TOTAL_FEATURES:>10d}   │ {best_meta_res['n_features']:>10d}           │
│ Feature Retention                 │     100.0%   │ {best_meta_res['feature_retention_pct']:>10.1f}%          │
│ Test Samples                      │ {len(msr_y_merged):>10d}   │ {len(msr_y_merged):>10d}           │
└───────────────────────────────────┴──────────────┴──────────────────────┘
""")

print("Cross-dataset evaluation complete!")
print(f"All results saved to: {CROSS_RESULTS}/")


INFERENCE TIMING SUMMARY (MSR-Action3D cross-dataset, 4-class)

┌───────────────────────────────────┬──────────────┬──────────────────────┐
│ Component                         │ Baseline     │ meta_WOA             │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Mask Application                  │     ---      │     0.0087 ms        │
│ NN Forward Pass                   │     0.1728 ms│     0.1543 ms        │
│ Total per-sample                  │     0.1826 ms│     0.1631 ms        │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Inference Speedup                 │    1.00x     │       1.12x          │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ MSR-Action3D Accuracy (4-class)   │     42.34%   │     57.66%           │
│ Random Chance (4-class)           │      25.0%   │      25.0%           │
│ Features Used                     │       1861   │        465           │
│ Feature Retention    